In [1]:
print("Hello")

Hello


In [2]:
!jupyter kernelspec list


0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
Available kernels:
  julia      /root/.local/share/jupyter/kernels/julia
  ir         /usr/local/share/jupyter/kernels/ir
  python3    /usr/local/share/jupyter/kernels/python3


In [3]:
!pip install torch

In [4]:
!pip install -U -q transformers datasets bitsandbytes trl peft huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 89.9 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 47.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 19.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 678.0/678.0 kB 53.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 642.6/642.6 kB 53.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 132.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 17.4 MB/s eta 0:00:00:00:0100:01


In [5]:
from huggingface_hub import notebook_login
notebook_login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Model

In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

model_name = "meta-llama/Llama-3.2-1B-Instruct"
config_8bit = BitsAndBytesConfig(load_in_8bit=True)
model_8bit = AutoModelForCausalLM.from_pretrained(model_name,
                                                  quantization_config=config_8bit,
                                                  device_map="auto",
                                                  dtype=torch.float16,
                                                  trust_remote_code=True,
                                                  )


config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Tokenizer

In [7]:
tokenizer = AutoTokenizer.from_pretrained(model_name, padding_side="left", trust_remote_code="True")

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Dataset

In [8]:
from datasets import load_dataset
dataset = load_dataset("MattCoddity/dockerNLcommands")
dataset

README.md: 0.00B [00:00, ?B/s]

06102023.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/2415 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input', 'output', 'instruction'],
        num_rows: 2415
    })
})

In [9]:
dataset["train"][0]

{'input': 'Give me a list of containers that have the Ubuntu image as their ancestor.',
 'output': "docker ps --filter 'ancestor=ubuntu'",
 'instruction': 'translate this sentence in docker command'}

In [10]:
from datasets import DatasetDict

train_test_split = dataset['train'].train_test_split(test_size=0.2
                                                     )
dataset = DatasetDict({
    'train': train_test_split['train'],
    'validation': train_test_split['test']
})
dataset

DatasetDict({
    train: Dataset({
        features: ['input', 'output', 'instruction'],
        num_rows: 1932
    })
    validation: Dataset({
        features: ['input', 'output', 'instruction'],
        num_rows: 483
    })
})

In [11]:
def to_chat_template(example):

    messages = [
        {"role": "system", "content": example['instruction']},
        {"role": "user", "content": example['input']},
        {"role": "assistant", "content": example['output']}
    ]

    
    return {'text':messages}

dataset = dataset.map(to_chat_template)
dataset

Map:   0%|          | 0/1932 [00:00<?, ? examples/s]

Map:   0%|          | 0/483 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input', 'output', 'instruction', 'text'],
        num_rows: 1932
    })
    validation: Dataset({
        features: ['input', 'output', 'instruction', 'text'],
        num_rows: 483
    })
})

In [12]:
dataset['train'][0]

{'input': 'List all the images along with their repository, tag, ID, and size, organizing them into a table.',
 'output': 'docker images --format "table {{.Repository}},{{.Tag}},{{.ID}},{{.Size}}"',
 'instruction': 'translate this sentence in docker command',
 'text': [{'content': 'translate this sentence in docker command',
   'role': 'system'},
  {'content': 'List all the images along with their repository, tag, ID, and size, organizing them into a table.',
   'role': 'user'},
  {'content': 'docker images --format "table {{.Repository}},{{.Tag}},{{.ID}},{{.Size}}"',
   'role': 'assistant'}]}

In [13]:
tokenizer.mistral = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.1")
tokenizer.mistral.apply_chat_template(dataset['train']['text'][0], tokenize=False)

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

'<s> [INST] translate this sentence in docker command\n\nList all the images along with their repository, tag, ID, and size, organizing them into a table. [/INST] docker images --format "table {{.Repository}},{{.Tag}},{{.ID}},{{.Size}}"</s>'

In [14]:
tokenizer.apply_chat_template(dataset['train']['text'][0],tokenize=False)

'<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 14 Apr 2026\n\ntranslate this sentence in docker command<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nList all the images along with their repository, tag, ID, and size, organizing them into a table.<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\ndocker images --format "table {{.Repository}},{{.Tag}},{{.ID}},{{.Size}}"<|eot_id|>'

In [15]:
def apply_chat_template(example):

    new_text = tokenizer.apply_chat_template(example['text'],tokenize=False)
    return {'text': new_text}

dataset = dataset.map(apply_chat_template)
dataset

Map:   0%|          | 0/1932 [00:00<?, ? examples/s]

Map:   0%|          | 0/483 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input', 'output', 'instruction', 'text'],
        num_rows: 1932
    })
    validation: Dataset({
        features: ['input', 'output', 'instruction', 'text'],
        num_rows: 483
    })
})

In [16]:
dataset['train']['text'][0]

'<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 14 Apr 2026\n\ntranslate this sentence in docker command<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nList all the images along with their repository, tag, ID, and size, organizing them into a table.<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\ndocker images --format "table {{.Repository}},{{.Tag}},{{.ID}},{{.Size}}"<|eot_id|>'

In [17]:
tokenizer(dataset['train']['text'][0])

{'input_ids': [128000, 128000, 128006, 9125, 128007, 271, 38766, 1303, 33025, 2696, 25, 6790, 220, 2366, 18, 198, 15724, 2696, 25, 220, 975, 5186, 220, 2366, 21, 271, 14372, 420, 11914, 304, 27686, 3290, 128009, 128006, 882, 128007, 271, 861, 682, 279, 5448, 3235, 449, 872, 12827, 11, 4877, 11, 3110, 11, 323, 1404, 11, 35821, 1124, 1139, 264, 2007, 13, 128009, 128006, 78191, 128007, 271, 29748, 5448, 1198, 2293, 330, 2048, 5991, 13, 4727, 39254, 3052, 13, 5786, 39254, 3052, 13, 926, 39254, 3052, 13, 1730, 24275, 128009], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [18]:
def tokenize_fn(example): 
    return tokenizer(example['text'])

tokenized_dataset = dataset.map(tokenize_fn, batched=True, remove_columns=['input', 'output', 'instruction', 'text'])
tokenized_dataset

Map:   0%|          | 0/1932 [00:00<?, ? examples/s]

Map:   0%|          | 0/483 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 1932
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 483
    })
})

In [19]:
len(tokenized_dataset['train']['input_ids'][1])

76

In [20]:
from transformers import DataCollatorForLanguageModeling
data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False, return_tensors = "pt")

In [21]:
tokenizer.pad_token

In [22]:
tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token

'<|eot_id|>'

#Data Loader

In [23]:

from torch.utils.data import DataLoader

data_loader = DataLoader(
    tokenized_dataset['train'],
    batch_size = 2,
    collate_fn = data_collator
)

for step, batch in enumerate(data_loader):
    print(f"Batch {step}")
    print("input_ids.shape: ", batch['input_ids'].shape)

    if step >= 3: 
        break

Batch 0
input_ids.shape:  torch.Size([2, 87])
Batch 1
input_ids.shape:  torch.Size([2, 83])
Batch 2
input_ids.shape:  torch.Size([2, 77])
Batch 3
input_ids.shape:  torch.Size([2, 96])


LoRa

In [24]:
from peft import LoraConfig, get_peft_model
import copy

model_8bit_clone = copy.deepcopy(model_8bit)

lora_config = LoraConfig(

    r = 32,
    lora_alpha = 16, #the higher value will be , lora matrix will get influence than matrix of base model
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "down_proj", "up_proj"],
    lora_dropout = 0.05,
    bias = "none", 
    task_type = "CAUSAL_LM"  
)

model_8bit_lora = get_peft_model(model_8bit_clone, lora_config)
model_8bit_lora.print_trainable_parameters()

trainable params: 22,544,384 || all params: 1,258,358,784 || trainable%: 1.7916


In [25]:
model_8bit

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear8bitLt(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear8bitLt(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear8bitLt(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear8bitLt(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear8bitLt(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear8bitLt(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear8bitLt(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm)

Hyper Traning

In [26]:

from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
                                    output_dir = "./results",
                                    per_device_train_batch_size = 2,
                                    per_device_eval_batch_size = 2,
                                    eval_steps = 10,
                                    eval_strategy = "steps",
                                    save_steps = 10,
                                    save_strategy = "steps",
                                    # num_train_epochs = 1,
                                    max_steps = 60,
                                    logging_steps = 10,
                                    learning_rate = 3e-5,
                                    weight_decay = 0.01,
                                    warmup_steps = 0,
                                    # fp16 = False,
                                    # bf16 = True,
                                    gradient_accumulation_steps = 4,
                                    optim = "adamw_torch",
                                    report_to = "none"
)


Training

In [27]:
tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 1932
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 483
    })
})

In [28]:

trainer = SFTTrainer(
    model = model_8bit_lora,
    train_dataset = tokenized_dataset['train'],
    eval_dataset = tokenized_dataset['validation'],
    data_collator = data_collator,
    # tokenizer = tokenizer,
    args = training_args,
)

trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009}.


Step,Training Loss,Validation Loss
10,4.278233,3.502103
20,3.181384,2.828264
30,2.606119,2.337665
40,2.184707,2.065540
50,1.961913,1.925471
60,1.898715,1.891165


TrainOutput(global_step=60, training_loss=2.685178597768148, metrics={'train_runtime': 310.274, 'train_samples_per_second': 1.547, 'train_steps_per_second': 0.193, 'total_flos': 224125947887616.0, 'train_loss': 2.685178597768148})

In [29]:
model_8bit_lora

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 2048)
        (layers): ModuleList(
          (0-15): 16 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear8bitLt(
                (base_layer): Linear8bitLt(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=32, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=32, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj):

LoRa with Trained Model 

In [30]:
import torch 
torch.set_printoptions(precision=8, sci_mode=False)

layer0 = model_8bit_lora.model.model.layers[0]

model_8bit_weights = layer0.self_attn.q_proj.weight

print("mddel_8bit_weights ", model_8bit_weights)

mddel_8bit_weights  Parameter containing:
Parameter(Int8Params([[-44,  16,  61,  ..., -22, -29,  50],
            [ 14,  70,  65,  ..., -39, -18,  13],
            [ 16,  14,  30,  ..., -34, -34, -24],
            ...,
            [ 17,  20,  40,  ..., -40, -15, -16],
            [ 32, -35,  50,  ..., -17, -41, -21],
            [-14, -30,  -7,  ...,  30,   5,  -2]], device='cuda:0',
           dtype=torch.int8))


In [33]:
import torch

torch.set_printoptions(precision=8, sci_mode=False)

layer0 = model_8bit_lora.model.model.layers[0]

weight_A = layer0.self_attn.q_proj.lora_A["default"].weight

print("LoRA A weights ", weight_A)

LoRA A weights  Parameter containing:
tensor([[-0.01312256, -0.00909424, -0.02197266,  ...,  0.01562500,
          0.00964355,  0.01672363],
        [-0.00521851,  0.01770020, -0.00878906,  ..., -0.00335693,
         -0.00427246,  0.00915527],
        [-0.01562500, -0.00537109,  0.01013184,  ..., -0.00866699,
         -0.01196289, -0.00692749],
        ...,
        [-0.01464844,  0.00781250, -0.01599121,  ...,  0.01525879,
          0.00854492, -0.01916504],
        [-0.00933838, -0.00296021, -0.00277710,  ...,  0.01953125,
         -0.00692749,  0.01330566],
        [ 0.01586914, -0.00213623,  0.01904297,  ..., -0.02014160,
         -0.00497437,  0.01403809]], device='cuda:0', dtype=torch.bfloat16,
       requires_grad=True)


Base Model

In [34]:
base_model = AutoModelForCausalLM.from_pretrained(
                                                   
                                                   model_name,
                                                   device_map="auto",
                                                   trust_remote_code=True
                                                   
)

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

In [ ]:
base_model 

#base_model is the orginal version of dataset

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (ro

In [36]:
import os

lora_path = "./lora_adapters"
model_8bit_lora.save_pretrained(lora_path)

In [37]:
from peft import PeftModel

base_model = PeftModel.from_pretrained(base_model, lora_path)
base_model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 2048)
        (layers): ModuleList(
          (0-15): 16 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=32, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=32, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear

Fully Implentation of LoRA

In [38]:
base_model = base_model.merge_and_unload()
base_model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (ro

Uploading model to huggingface
